# European options

In [1]:
# We can pass functions as arguments

def adder(a, b):
    return a+b

def multiplier(a, b):
    return a*b

def arithmetic(a, b, some_func):
    return some_func(a, b)

In [2]:
print(arithmetic(2, 5, multiplier))

10


In [3]:
import math

def call_payoff(stock, strike):
    """
    Payoff for holding a call option with a given stock price and strike price
    """
    return max(stock-strike, 0)


def put_payoff(stock, strike):
    """
    Payoff for holding a put option with a given stock price and strike price
    """
    return max(strike-stock, 0)


def european_option_pricer(stock_init, strike, U, D, R, N, payoff_func):
    """
    Give the price of a call or put using a version of Cox-Ross-Rubinstein.

    Args:
        stock_init (float): price of the asset right now
        strike (float): strike price of the option
        U (float): The up percentage, per step, as a decimal (0 < U < 1)
        D (float): The down percentage, per step, as a decimal (0 < D < 1)
        R (float): The risk-free interest in one step, as a decimal (D < R < U)
        N (int): Number of steps
        payoff_func (function): This is a function that will take a stock price
            and a strike price.

    Returns:
        option_price (float). Returns option price using a binomial model.
    """
    option_price = 0

    # Let's compute p* as this will be the same throughout the computation
    p_star = (R-D)/(U-D)

    # Be cognizant of computer ranges
    for i in range(N+1):

        # Compute the stock price after i ups and N-i downs, and the payoff
        stock_price = stock_init * (1+U)**i * (1+D)**(N-i) 
        payoff = payoff_func(stock_price, strike)

        # Compute the combinatorial term.  Note that here we are counting on
        # the payoff function to contribute zeros where appropriate.
        term = math.comb(N,i) * p_star**i * (1-p_star)**(N-i) * payoff
        option_price += term

    return option_price/((1+R)**N)

In [4]:
european_option_pricer(100, 100, 0.1, -0.1, 0.05, 1, call_payoff)

7.142857142857154

In [5]:
# Hand check
res = (1/1.05)*(0.75*10 + 0.25*0)
print(res)

7.142857142857142


In [6]:
european_option_pricer(100, 105, 0.12, 0.02, 105/95-1, 1, put_payoff)

0.39999999999999697

In [7]:
105/95-1

0.10526315789473695

In [8]:
0.7*((3199-2865)/2865)**2 + 0.3*((2979-2865)/2865)**2 - 0.09354275741710297**2

0.00123827014975832

# Recursion
### Fibonacci

$F_1 = 1$

$F_2 = 1$

$F_n = F_{n-1} + F_{n-2}   \;\;\;\;\;\;\; n \geq 3$

In [9]:
# Recursive definitions can use recursive functions

# from functools import cache

# @cache
def fib(n):
    if n==1 or n==2:
        return 1
    return fib(n-1) + fib(n-2)

In [10]:
"""
We can model option pricing recursively.
The closed formula for European options is probably preferable, for both perfomance and simplicity.
"""

def recursive_european_option_pricer(S, K, U, D, R, N, payoff_func):
    
    # Our base step, if we are at a 0-step tree, return the payoff
    if N == 0:
        return payoff_func(S, K)
    
    # Compute the risk-neutral probability
    p = (R-D)/(U-D)
    
    return 1/(1+R) * (
        p * recursive_european_option_pricer(S*(1+U), K, U, D, R, N-1, payoff_func)
        +
        (1-p) * recursive_european_option_pricer(S*(1+D), K, U, D, R, N-1, payoff_func)
    )


In [11]:
print(recursive_european_option_pricer(100, 100, 0.1, -0.1, 0.05, 1, call_payoff))
print(recursive_european_option_pricer(100, 100, 0.1, -0.1, 0.05, 3, put_payoff))

7.142857142857154
1.6898823021271971


# American Options

In [12]:
def american_option_pricer(S, K, U, D, R, N, payoff_func):

    if N == 0:
        return payoff_func(S, K)
    
    p = (R-D)/(U-D)
    
    # discounted risk-neutral expected value
    discounted_EV = 1/(1+R) * (
        p * american_option_pricer(S*(1+U), K, U, D, R, N-1, payoff_func)
        +
        (1-p) * american_option_pricer(S*(1+D), K, U, D, R, N-1, payoff_func)
    )
    return max(payoff_func(S,K), discounted_EV)

In [13]:
print(american_option_pricer(100, 100, 0.1, -0.1, 0.05, 3, put_payoff))

2.8223194039520556


In [23]:
from collections import defaultdict


U = 0.1
D = -0.05
R = 0.05
p = (R-D)/(U-D)

N = 3
S = 100
K = 100

def put_payoff(S,K):
    return max(K-S, 0)

def call_payoff(S,K):
    return max(S-K, 0)

tree = defaultdict(lambda: defaultdict(dict))

print("STEP\tUP #\tDOWN #\tPRICE\tOPTION\tE*")
for step in range(N, -1, -1):
    print()
    for j in range(step, -1, -1):
        price = S * (1+U)**j * (1+D)**(step - j)
        payoff = put_payoff(price, K)
        if step != N:
            exp_val = (1/(1+R)) * (p * tree[step + 1][j+1]["option"] + (1-p) * tree[step + 1][j]["option"])
        else:
            exp_val = payoff
        option = max(payoff, exp_val)
        tree[step][j] = {
            "price": price,
            "option": option,
            "expected_val": exp_val
        }
        
        print(f"{step}\t{j}\t{step-j}\t{price:.2f}\t{option:.2f}\t({exp_val:.2f})")

def print_tree(T):
    N = len(T)
    dim = 2*N-1
    arr = []
    for i in range(2*dim):
        arr.append(dim*[" "*8])
        
    for step, ups in T.items():
        for up, data in ups.items():
            p = "{:8.2f}".format(data["price"])
            o = "{:8.2f}".format(data["option"])
            x = 2*step
            y_p = 4*(N-up-1) - 2*(N-step-1)

            arr[y_p][x] = p
            arr[y_p+1][x] = o
    
    for ln in arr:
        print("".join(ln))
    
print_tree(tree)

STEP	UP #	DOWN #	PRICE	OPTION	E*

3	3	0	133.10	0.00	(0.00)
3	2	1	114.95	0.00	(0.00)
3	1	2	99.28	0.72	(0.72)
3	0	3	85.74	14.26	(14.26)

2	2	0	121.00	0.00	(0.00)
2	1	1	104.50	0.23	(0.23)
2	0	2	90.25	9.75	(4.99)

1	1	0	110.00	0.07	(0.07)
1	0	1	95.00	5.00	(3.24)

0	0	0	100.00	1.63	(1.63)
                                                  133.10
                                                    0.00
                                  121.00                
                                    0.00                
                  110.00                          114.95
                    0.07                            0.00
  100.00                          104.50                
    1.63                            0.23                
                   95.00                           99.28
                    5.00                            0.72
                                   90.25                
                                    9.75                
                               